# Notebook 2: Data Cleaning

This notebook prepares the raw datasets for later integration and analysis. The cleaning process includes standardising variable names, correcting data types, handling disclosure-controlled entries, selecting relevant variables and checking the quality of the processed datasets.

The original source files remain unchanged. Cleaned datasets will be saved separately in the processed data folder.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")

Libraries imported successfully.


## Project Folders and Dataset Locations

The project folders and raw dataset locations are defined to ensure that the cleaning process can be reproduced without changing the original files.

In [2]:
project_folder = Path.cwd().parent

raw_folder = project_folder / "Datasets" / "Raw"
processed_folder = project_folder / "Datasets" / "processed"

processed_folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)
print("Raw folder exists:", raw_folder.exists())
print("Processed folder exists:", processed_folder.exists())

Project folder: /Users/adityakumar/Desktop/EV_Dissertation
Raw folder exists: True
Processed folder exists: True


In [3]:
ev_file = raw_folder / "df_VEH0145.csv"

lookup_file = (
    raw_folder
    / "Lower_Layer_Super_Output_Area_(2021)_to_LAD_(April_2023)_Lookup_in_England_and_Wales.csv"
)

charging_file = (
    raw_folder
    / "evci9001_2026-04_EV_charging_devices_UK.ods"
)

population_file = raw_folder / "sapelsoasyoa20222024.xlsx"

## Load the Raw Datasets

The datasets are loaded using the header positions identified during the initial inspection. Only the local-authority charging table and the mid-2024 population table are required at this stage.

In [4]:
ev_data = pd.read_csv(
    ev_file,
    low_memory=False
)

lookup_data = pd.read_csv(
    lookup_file,
    low_memory=False
)

charging_data = pd.read_excel(
    charging_file,
    sheet_name="1a",
    engine="odf",
    header=2
)

population_data = pd.read_excel(
    population_file,
    sheet_name="Mid-2024 LSOA 2021",
    header=3
)

print("EV data:", ev_data.shape)
print("Lookup data:", lookup_data.shape)
print("Charging data:", charging_data.shape)
print("Population data:", population_data.shape)

EV data: (285505, 62)
Lookup data: (35672, 7)
Charging data: (433, 29)
Population data: (35672, 187)


## Clean the Electric Vehicle Dataset

The EV registration dataset is prepared for later analysis by standardising the variable names, identifying the quarterly registration columns and converting the quarterly values into a numeric format. Disclosure-controlled values published by the data provider are treated as missing values to preserve data integrity.

In [5]:
# Create a working copy of the EV dataset

ev_clean = ev_data.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [6]:
# Remove unnecessary spaces from column names and text fields

ev_clean.columns = ev_clean.columns.str.strip()

text_columns = [
    "LSOA21CD",
    "LSOA21NM",
    "Fuel",
    "Keepership"
]

for column in text_columns:
    ev_clean[column] = (
        ev_clean[column]
        .astype(str)
        .str.strip()
    )

print("Text fields standardised successfully.")

Text fields standardised successfully.


In [7]:
# Identify the quarterly registration columns

quarter_columns = [
    column
    for column in ev_clean.columns
    if re.fullmatch(r"\d{4} Q[1-4]", str(column))
]

print("Number of quarter columns:", len(quarter_columns))
print("First quarter:", quarter_columns[-1])
print("Latest quarter:", quarter_columns[0])

Number of quarter columns: 58
First quarter: 2011 Q4
Latest quarter: 2026 Q1


In [8]:
# Identify non-numeric entries in the quarterly columns

non_numeric_counts = {}

for column in quarter_columns:
    values = ev_clean[column].astype(str).str.strip()

    numeric_mask = values.str.fullmatch(
        r"-?\d+(\.\d+)?",
        na=False
    )

    non_numeric_values = values[
        ~numeric_mask & values.ne("nan")
    ]

    for value, count in non_numeric_values.value_counts().items():
        non_numeric_counts[value] = (
            non_numeric_counts.get(value, 0) + int(count)
        )

non_numeric_counts

{'[c]': 4814923, '[x]': 65065}

In [9]:
# Replace disclosure-controlled entries with missing values

ev_clean[quarter_columns] = (
    ev_clean[quarter_columns]
    .replace({
        "[c]": np.nan,
        "[x]": np.nan
    })
)

print("Disclosure-controlled entries replaced successfully.")

Disclosure-controlled entries replaced successfully.


In [10]:
# Convert quarterly registration columns to numeric values

ev_clean[quarter_columns] = (
    ev_clean[quarter_columns]
    .apply(pd.to_numeric, errors="coerce")
)

print("Quarterly columns converted successfully.")

Quarterly columns converted successfully.


In [11]:
# Check the data types of the quarterly columns

ev_clean[quarter_columns].dtypes.value_counts()

float64    58
Name: count, dtype: int64

### Cleaning Outcome

The disclosure-controlled entries were successfully converted to missing values before the quarterly registration columns were transformed into numeric variables. This ensures that suppressed values are not interpreted as genuine vehicle registrations while allowing the dataset to be used for statistical analysis and time-series modelling.

## Clean the Geography Lookup Dataset

The geography lookup dataset links Lower Layer Super Output Areas (LSOAs) to Local Authority Districts (LADs). Only the variables required for the analysis are retained, while unnecessary columns are removed. The geographical identifiers are also standardised to ensure consistency across all datasets.

In [12]:
# Select the required columns

lookup_clean = lookup_data[
    [
        "LSOA21CD",
        "LSOA21NM",
        "LAD23CD",
        "LAD23NM"
    ]
].copy()

lookup_clean.head()

,LSOA21CD,LSOA21NM,LAD23CD,LAD23NM
0,E01011949,Hartlepool 009A,E06000001,Hartlepool
1,E01011950,Hartlepool 008A,E06000001,Hartlepool
2,E01011951,Hartlepool 007A,E06000001,Hartlepool
3,E01011952,Hartlepool 002A,E06000001,Hartlepool
4,E01011953,Hartlepool 002B,E06000001,Hartlepool


In [13]:
# Rename the columns

lookup_clean = lookup_clean.rename(
    columns={
        "LSOA21CD": "lsoa_code",
        "LSOA21NM": "lsoa_name",
        "LAD23CD": "lad_code",
        "LAD23NM": "lad_name"
    }
)

lookup_clean.head()

,lsoa_code,lsoa_name,lad_code,lad_name
0,E01011949,Hartlepool 009A,E06000001,Hartlepool
1,E01011950,Hartlepool 008A,E06000001,Hartlepool
2,E01011951,Hartlepool 007A,E06000001,Hartlepool
3,E01011952,Hartlepool 002A,E06000001,Hartlepool
4,E01011953,Hartlepool 002B,E06000001,Hartlepool


In [14]:
# Standardise the text fields

text_columns = [
    "lsoa_code",
    "lsoa_name",
    "lad_code",
    "lad_name"
]

for column in text_columns:
    lookup_clean[column] = (
        lookup_clean[column]
        .astype(str)
        .str.strip()
    )

print("Text fields standardised successfully.")

Text fields standardised successfully.


In [15]:
# Check the dataset quality

print("Rows:", len(lookup_clean))
print("Unique LSOA codes:", lookup_clean["lsoa_code"].nunique())
print("Duplicate LSOA codes:", lookup_clean["lsoa_code"].duplicated().sum())
print("Missing values:", lookup_clean.isna().sum().sum())

Rows: 35672
Unique LSOA codes: 35672
Duplicate LSOA codes: 0
Missing values: 0


### Cleaning Outcome

The geography lookup dataset was reduced to the variables required for this study. The geographical identifiers were standardised by removing unnecessary spaces, and no duplicate or missing geographical codes were identified. The processed lookup table provides a reliable link between LSOAs and Local Authority Districts for the later integration stage.

## Clean the Charging Infrastructure Dataset

The charging infrastructure dataset contains quarterly counts of publicly available charging devices by geographical area. The cleaning process standardises the column names, removes spreadsheet note references and converts the quarterly charging counts into numeric variables.

In [16]:
# Create a working copy

charging_clean = charging_data.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [17]:
# Standardise the column names

charging_clean.columns = (
    charging_clean.columns
    .astype(str)
    .str.replace(r"\s*\[Note\s*\d+\]", "", regex=True)
    .str.strip()
)

charging_clean.columns.tolist()

['Local authority / region code',
 'Local authority / region name',
 'Oct-19',
 'Jan-20',
 'Apr-20',
 'Jul-20',
 'Oct-20',
 'Jan-21',
 'Apr-21',
 'Jul-21',
 'Oct-21',
 'Jan-22',
 'Apr-22',
 'Jul-22',
 'Oct-22',
 'Jan-23',
 'Apr-23',
 'Jul-23',
 'Oct-23',
 'Jan-24',
 'Apr-24',
 'Jul-24',
 'Oct-24',
 'Jan-25',
 'Apr-25',
 'Jul-25',
 'Oct-25',
 'Jan-26',
 'Apr-26']

In [18]:
# Rename the geographical columns

charging_clean = charging_clean.rename(
    columns={
        charging_clean.columns[0]: "area_code",
        charging_clean.columns[1]: "area_name"
    }
)

charging_clean.head()

,area_code,area_name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,Oct-21,Jan-22,Apr-22,Jul-22,Oct-22,Jan-23,Apr-23,Jul-23,Oct-23,Jan-24,Apr-24,Jul-24,Oct-24,Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26
0,K02000001,United Kingdom,15116,16505,17947,18265,19487,20775,22790,24374,25927,28375,30290,32011,34637,37055,40150,44020,49220,53677,59670,64632,70042,73334,76507,82002,86021,87796,92141
1,K03000001,Great Britain,14821,16210,17642,17953,19169,20455,22463,24044,25595,28030,29942,31683,34295,36689,39761,43587,48789,53212,59125,64014,69403,72654,75835,81318,85283,87069,91414
2,E92000001,England,12549,13719,14979,15395,16456,17459,19261,20563,21925,24159,25884,27502,29774,31466,34203,37717,42489,46374,51506,55631,60364,63389,65877,70672,74115,75818,79198
3,E12000001,North East,738,752,786,812,849,820,854,887,916,975,1011,1155,1142,1253,1392,1657,1536,1596,1942,1919,2295,2583,2553,2703,2698,2734,2950
4,E06000047,County Durham,92,96,102,105,106,110,121,116,124,128,149,174,206,229,240,259,268,291,309,313,359,400,462,482,482,489,523


In [19]:
# Standardise the geographical fields

charging_clean["area_code"] = (
    charging_clean["area_code"]
    .astype(str)
    .str.strip()
)

charging_clean["area_name"] = (
    charging_clean["area_name"]
    .astype(str)
    .str.strip()
)

print("Geographical fields standardised successfully.")

Geographical fields standardised successfully.


In [20]:
# Identify the quarterly charging columns

charging_quarter_columns = list(charging_clean.columns[2:])

print("Number of quarter columns:", len(charging_quarter_columns))

charging_quarter_columns

Number of quarter columns: 27


['Oct-19',
 'Jan-20',
 'Apr-20',
 'Jul-20',
 'Oct-20',
 'Jan-21',
 'Apr-21',
 'Jul-21',
 'Oct-21',
 'Jan-22',
 'Apr-22',
 'Jul-22',
 'Oct-22',
 'Jan-23',
 'Apr-23',
 'Jul-23',
 'Oct-23',
 'Jan-24',
 'Apr-24',
 'Jul-24',
 'Oct-24',
 'Jan-25',
 'Apr-25',
 'Jul-25',
 'Oct-25',
 'Jan-26',
 'Apr-26']

In [21]:
# Convert the quarterly charging columns to numeric values

charging_clean[charging_quarter_columns] = (
    charging_clean[charging_quarter_columns]
    .apply(pd.to_numeric, errors="coerce")
)

charging_clean[charging_quarter_columns].dtypes.value_counts()

float64    27
Name: count, dtype: int64

In [22]:
# Check the dataset quality

print("Rows:", len(charging_clean))
print("Duplicate rows:", charging_clean.duplicated().sum())
print("Missing values:", charging_clean.isna().sum().sum())

Rows: 433
Duplicate rows: 0
Missing values: 474


In [23]:
# Check which columns contain missing values

missing_summary = (
    charging_clean
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

Apr-26    28
Jan-26    28
Oct-25    28
Jul-25    28
Apr-25    28
Jan-25    28
Oct-24    28
Jul-24    28
Apr-24    28
Jan-24    28
Oct-23    28
Jul-23    28
Jan-23    12
Apr-23    12
Oct-22    12
Jul-22    12
Apr-22    12
Jan-22    12
Oct-21    12
Jul-21    12
Apr-21     6
Jan-21     6
Oct-20     6
Jul-20     6
Apr-20     6
Jan-20     6
Oct-19     6
dtype: int64

In [24]:
# Display only the geographical information for rows with missing values

charging_clean.loc[
    charging_clean.isna().any(axis=1),
    ["area_code", "area_name"]
]

,area_code,area_name
22,E06000063,Cumberland (from April 2023)
25,E06000064,Westmorland and Furness (from April 2023)
26,E10000006,Cumbria (abolished April 2023)
27,E07000026,Allerdale (abolished April 2023)
28,E07000027,Barrow-in-Furness (abolished April 2023)
29,E07000028,Carlisle (abolished April 2023)
30,E07000029,Copeland (abolished April 2023)
31,E07000030,Eden (abolished April 2023)
32,E07000031,South Lakeland (abolished April 2023)
68,E06000065,North Yorkshire (from April 2023)


### Missing Values Assessment

The missing values identified in the charging infrastructure dataset are not caused by data quality issues. They correspond to local authorities that were abolished following UK local government boundary changes (for example, the 2021 and 2023 unitary authority reorganisations). Consequently, charging device statistics are unavailable for these authorities after the administrative changes. These missing values were retained because they represent genuine absence of published data rather than errors in data collection.

## Clean the Population Dataset

The population dataset contains total population estimates and detailed age-by-sex variables for each LSOA. Only the geographical identifiers and total population are required for this study. The remaining demographic variables are outside the scope of the research objectives and are excluded from the processed dataset.

In [25]:
# Select the variables required for the analysis

population_clean = population_data[
    [
        "LAD 2023 Code",
        "LAD 2023 Name",
        "LSOA 2021 Code",
        "LSOA 2021 Name",
        "Total"
    ]
].copy()

In [26]:
# Rename the population variables

population_clean = population_clean.rename(
    columns={
        "LAD 2023 Code": "lad_code",
        "LAD 2023 Name": "lad_name",
        "LSOA 2021 Code": "lsoa_code",
        "LSOA 2021 Name": "lsoa_name",
        "Total": "population_2024"
    }
)

In [27]:
# Standardise the geographical fields

population_text_columns = [
    "lad_code",
    "lad_name",
    "lsoa_code",
    "lsoa_name"
]

for column in population_text_columns:
    population_clean[column] = (
        population_clean[column]
        .astype(str)
        .str.strip()
    )

population_clean["population_2024"] = pd.to_numeric(
    population_clean["population_2024"],
    errors="coerce"
)

print("Population dataset cleaned successfully.")

Population dataset cleaned successfully.


In [28]:
# Check the cleaned population dataset

print("Rows:", len(population_clean))
print(
    "Unique LSOA codes:",
    population_clean["lsoa_code"].nunique()
)
print(
    "Duplicate LSOA codes:",
    population_clean["lsoa_code"].duplicated().sum()
)
print(
    "Missing population values:",
    population_clean["population_2024"].isna().sum()
)

Rows: 35672
Unique LSOA codes: 35672
Duplicate LSOA codes: 0
Missing population values: 0


In [29]:
population_clean.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population_2024
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


## Save the Cleaned Datasets

The cleaned datasets are saved in the processed data folder for use during the data integration stage. The original raw files remain unchanged.

In [30]:
# Save the cleaned datasets

ev_clean.to_csv(
    processed_folder / "ev_clean.csv",
    index=False
)

charging_clean.to_csv(
    processed_folder / "charging_clean.csv",
    index=False
)

lookup_clean.to_csv(
    processed_folder / "lookup_clean.csv",
    index=False
)

population_clean.to_csv(
    processed_folder / "population_clean.csv",
    index=False
)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.
